In [0]:
%scala
case class Flight(DEST_COUNTRY_NAME: String, ORIGIN_COUNTRY_NAME: String, count: BigInt) 


defined class Flight

In [0]:
%scala

val flightsDF = spark.read
  .option("header", "true") // jeśli plik CSV ma nagłówki
  .option("inferSchema", "true") // opcjonalnie: automatyczne rozpoznawanie typów
  .csv("/FileStore/tables/2010_summary-2.csv")


flightsDF: org.apache.spark.sql.DataFrame = [DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string ... 1 more field]

In [0]:
%scala
val flightsDS = flightsDF.as[Flight]


flightsDS: org.apache.spark.sql.Dataset[Flight] = [DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string ... 1 more field]

In [0]:
%scala
display(flightsDS.select($"DEST_COUNTRY_NAME", $"ORIGIN_COUNTRY_NAME",$"count"))

DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count
United States,Romania,1
United States,Ireland,264
United States,India,69
Egypt,United States,24
Equatorial Guinea,United States,1
United States,Singapore,25
United States,Grenada,54
Costa Rica,United States,477
Senegal,United States,29
United States,Marshall Islands,44


In [0]:
%scala
import org.apache.spark.sql.{DataFrame, Dataset, SparkSession}
import spark.implicits._
import org.apache.spark.sql.functions._
// Wykonaj mnożenie kolumny DEST_COUNTRY_NAME * string używając withColumn i zobacz co się stanie

val dfIncreased = flightsDS.withColumn("new_col", col("DEST_COUNTRY_NAME") * "abc")


display(dfIncreased)

DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count,new_col
United States,Romania,1,null
United States,Ireland,264,null
United States,India,69,null
Egypt,United States,24,null
Equatorial Guinea,United States,1,null
United States,Singapore,25,null
United States,Grenada,54,null
Costa Rica,United States,477,null
Senegal,United States,29,null
United States,Marshall Islands,44,null


In [0]:
%scala

// Wykonaj mnożenie kolumny DEST_COUNTRY_NAME * string używając funkcji map() i zobacz co się stanie
val dsError = flightsDS.map(row =>
  row.copy(DEST_COUNTRY_NAME = row.DEST_COUNTRY_NAME * 2)
)

display(dsError)


DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count
United StatesUnited States,Romania,1
United StatesUnited States,Ireland,264
United StatesUnited States,India,69
EgyptEgypt,United States,24
Equatorial GuineaEquatorial Guinea,United States,1
United StatesUnited States,Singapore,25
United StatesUnited States,Grenada,54
Costa RicaCosta Rica,United States,477
SenegalSenegal,United States,29
United StatesUnited States,Marshall Islands,44


In [0]:
%scala
def matchFields(row: Flight): Boolean = {
  return row.DEST_COUNTRY_NAME == row.ORIGIN_COUNTRY_NAME
}


matchFields: (row: Flight)Boolean

In [0]:
%scala
// Użyj funkcji matchFields na Datafram zaobserwuj co się dzieje
flightsDF.filter(row => matchFields(row)).show()

In [0]:
%scala
// Użyj funkcji matchFields na Dataset 
flightsDS.filter(row => matchFields(row)).show()

+-----------------+-------------------+------+
DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME| count|
+-----------------+-------------------+------+
 United States| United States|348113|
+-----------------+-------------------+------+

In [0]:
%scala
case class FlightMetadata(count: BigInt ,randomData: Int)


defined class FlightMetadata

In [0]:
%scala
val flightsMeta = spark.range(500).map(x => (x, scala.util.Random.nextInt))
.withColumnRenamed("_1","count")
.withColumnRenamed("_2","randomData").as[FlightMetadata]

display(flightsMeta)

count,randomData
0,2010649913
1,1147005694
2,447352960
3,-103383423
4,607079239
5,-1394928200
6,98735869
7,594463834
8,-163536954
9,403884948


In [0]:
%scala

// Wykonaj join pomiędzy flightsMeta i flightsDF po kolumnie 'count'
val flights2 = flightsDF.join(flightsMeta, Seq("count"))
display(flights2)

count,DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,randomData
1,United States,Romania,-855494321
264,United States,Ireland,-42198589
69,United States,India,245553165
24,Egypt,United States,1622626931
1,Equatorial Guinea,United States,-855494321
25,United States,Singapore,-1341225551
54,United States,Grenada,1482632586
477,Costa Rica,United States,2031343407
29,Senegal,United States,-440779261
44,United States,Marshall Islands,1025416663


Joins

In [0]:
%scala

// Wykonaj joinWith pomiędzy flightsMeta i flightsDF po kolumnie 'count'
val flights3 = flightsDS.joinWith(flightsMeta, flightsDS("count") === flightsMeta("count"))
display(flights3)


_1,_2
"List(United States, Romania, 1)","List(1, -96789884)"
"List(United States, Ireland, 264)","List(264, 1466184142)"
"List(United States, India, 69)","List(69, -948711429)"
"List(Egypt, United States, 24)","List(24, 636065846)"
"List(Equatorial Guinea, United States, 1)","List(1, -96789884)"
"List(United States, Singapore, 25)","List(25, -232593988)"
"List(United States, Grenada, 54)","List(54, 1674699692)"
"List(Costa Rica, United States, 477)","List(477, -7204154)"
"List(Senegal, United States, 29)","List(29, 2107692785)"
"List(United States, Marshall Islands, 44)","List(44, 1811610139)"


In [0]:
%scala
// Wykonaj funkcję groupBy na dataset po kolumnie "DEST_COUNTRY_NAME" i zobacz jaki stworzy się obiekt
val grouped = flightsDS.groupBy("DEST_COUNTRY_NAME")
grouped // <- to zwraca RelationalGroupedDataset



grouped: org.apache.spark.sql.RelationalGroupedDataset = RelationalGroupedDataset: [grouping expressions: [DEST_COUNTRY_NAME: string], value: [DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string ... 1 more field], type: GroupBy]
res14: org.apache.spark.sql.RelationalGroupedDataset = RelationalGroupedDataset: [grouping expressions: [DEST_COUNTRY_NAME: string], value: [DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string ... 1 more field], type: GroupBy]

In [0]:
%scala
// Podlicz ile było lotów z 'DEST_COUNTRY_NAME' używając funkcji groupByKey
display(
  flightsDS
    .map(flight => (flight.DEST_COUNTRY_NAME, 1))
    .groupByKey(_._1)
    .mapGroups { case (country, iter) => (country, iter.size) }
)



_1,_2
Afghanistan,1
Angola,1
Anguilla,1
Antigua and Barbuda,1
Argentina,1
Aruba,1
Australia,1
Austria,1
Azerbaijan,1
Bahrain,1
